# Redis: Topics

1. What Redis is and why it's used
2. How to connect to Redis from Python
3. All the core **data types** (strings, hashes, lists, sets, sorted sets)
4. **Key expiration** (TTL) — the foundation of caching
5. **Pub/Sub** messaging
6. **Pipelines & transactions** (performance)
7. A **mini real-world project**: a simple cache + leaderboard

> 📌 Just read top-to-bottom and run each cell. Every concept builds on the previous one.

## 1. What is Redis?

**Redis** = **RE**mote **DI**ctionary **S**erver.

- An **in-memory** key-value data store (data lives in RAM → very fast, sub-millisecond).
- Think of it like a giant Python dictionary that:
  - Lives on a server (so multiple apps can share it)
  - Survives restarts (optional persistence)
  - Supports rich data types beyond plain strings
- Common uses: **caching**, **session storage**, **rate limiting**, **leaderboards**, **queues**, **pub/sub messaging**.

### Mental model
```
Your App  ──TCP──>  Redis Server  ──>  In-memory key/value store
                       (port 6379)
```

## 2. Setup — Get Redis Running

You need **two things**:

### a) A running Redis server
Easiest option (Windows-friendly): **Docker**
```powershell
docker run -d --name redis -p 6379:6379 redis
```
Other options: WSL (`sudo apt install redis-server`), Memurai (Windows native), or a free cloud Redis on [redis.io](https://redis.io).

### b) The Python client
```powershell
pip install redis
```
(Already listed in `requirements.txt` in this repo.)

In [14]:
print("Hello")

Hello


In [15]:
# Install the Python client (uncomment if not yet installed)
# %pip install redis

import redis

# Connect to Redis running on localhost:6379
# decode_responses=True  -> return Python str instead of bytes (nicer for beginners)
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)

# A quick health check — PING should return "PONG"
print("Server says:", r.ping())

Server says: True


## 3. The Big Idea: Keys and Values

Everything in Redis is stored under a **key** (a string). The **value** can be one of several types:

| Type        | Looks like                            | Use case                          |
|-------------|---------------------------------------|-----------------------------------|
| String      | `"hello"` / `42`                      | Cached values, counters           |
| Hash        | `{name: "Alice", age: 30}`            | Objects / records                 |
| List        | `["a", "b", "c"]` (ordered)           | Queues, recent items              |
| Set         | `{"a", "b", "c"}` (unique, unordered) | Tags, unique visitors             |
| Sorted Set  | `{a: 1.0, b: 2.5}` (sorted by score)  | Leaderboards, priority queues     |
| Stream      | append-only log                       | Event sourcing, messaging         |

**Naming convention:** people use colons to namespace keys, e.g. `user:1:name`, `session:abc123`.

## 4. Strings — The Simplest Type

A "string" in Redis can hold text, numbers, or even binary data (up to 512 MB).

Key commands:
- `SET key value` — store
- `GET key` — fetch
- `DEL key` — delete
- `INCR key` / `DECR key` — atomic counter
- `EXISTS key` — check if exists

In [16]:
# --- Basic SET / GET ---
r.set("greeting", "Hello, Redis!")
print("greeting =", r.get("greeting"))

# --- Counter (great for tracking views, likes, etc.) ---
r.set("page:views", 0)        # initialize
r.incr("page:views")          # +1  -> 1
r.incr("page:views")          # +1  -> 2
r.incrby("page:views", 10)    # +10 -> 12
print("page:views =", r.get("page:views"))

# --- Existence / deletion ---
print("greeting exists?", r.exists("greeting"))
r.delete("greeting")
print("greeting after delete:", r.get("greeting"))   # None

greeting = Hello, Redis!
page:views = 12
greeting exists? 1
greeting after delete: None


In [17]:
r.set("temp", "This will expire in 5 seconds")
r.expire("temp", 5)  # Set a TTL of 5 seconds
print("temp exists?", r.exists("temp"))
import time
time.sleep(6)  # Wait for 6 seconds
print("temp exists after 6 seconds?", r.exists("temp"))

temp exists? 1
temp exists after 6 seconds? 0


## 5. Key Expiration (TTL) — The Heart of Caching

You can tell Redis to **automatically delete** a key after N seconds. This is what makes Redis perfect as a cache.

- `SET key value EX seconds` — set with expiry
- `EXPIRE key seconds` — add expiry to existing key
- `TTL key` — seconds remaining (-1 = no expiry, -2 = key gone)

In [18]:
import time

# Set a key that auto-expires in 5 seconds
r.set("temp:otp", "123456", ex=5)
print("ttl right now:", r.ttl("temp:otp"), "sec")
print("value:", r.get("temp:otp"))

time.sleep(6)  # wait for expiry
print("ttl after 6s :", r.ttl("temp:otp"))   # -2 means: key no longer exists
print("value after  :", r.get("temp:otp"))   # None

ttl right now: 5 sec
value: 123456
ttl after 6s : -2
value after  : None


## 6. Hashes — Storing Objects

A **hash** is a key whose value is a small dictionary of field → value. Perfect for storing records like a user.

- `HSET key field value [field value ...]`
- `HGET key field`
- `HGETALL key` — get the whole dict
- `HINCRBY key field n` — atomic increment of one field

In [19]:
# Store a user record under key "user:1"
r.hset("user:1", mapping={
    "name": "Alice",
    "email": "alice@example.com",
    "age": 30,
})

print("name :", r.hget("user:1", "name"))
print("all  :", r.hgetall("user:1"))

# Atomically bump a numeric field
r.hincrby("user:1", "age", 1)
print("age++:", r.hget("user:1", "age"))

name : Alice
all  : {'name': 'Alice', 'email': 'alice@example.com', 'age': '30'}
age++: 31


In [20]:
import json

# Redis strings only accept bytes/str/int/float — serialize the dict first.
user = {"name": "Bob", "email": "bob@example.com"}
r.set("user:data", json.dumps(user))

raw = r.get("user:data")          # str (because decode_responses=True)
parsed = json.loads(raw)          # back to a real dict

print("raw    :", raw, type(raw).__name__)
print("parsed :", parsed, type(parsed).__name__)
print("email  :", parsed["email"])

raw    : {"name": "Bob", "email": "bob@example.com"} str
parsed : {'name': 'Bob', 'email': 'bob@example.com'} dict
email  : bob@example.com


## 7. Lists — Ordered Sequences (Queues)

A list is like a Python list, but optimized for pushing/popping at the ends.

- `LPUSH key v` — push to the **left** (head)
- `RPUSH key v` — push to the **right** (tail)
- `LPOP` / `RPOP` — pop from either end
- `LRANGE key start stop` — read a range (use `0 -1` for "everything")

A common pattern: **`LPUSH` + `RPOP` = a queue** (FIFO).

In [21]:
# Clear any leftover list from previous runs
r.delete("tasks")

# Producers add tasks on the left
r.lpush("tasks", "send-email")
r.lpush("tasks", "resize-image")
r.lpush("tasks", "generate-report")

# View the whole list
print("tasks:", r.lrange("tasks", 0, -1))

# A worker processes from the right (FIFO)
print("processing:", r.rpop("tasks"))
print("processing:", r.rpop("tasks"))
print("remaining :", r.lrange("tasks", 0, -1))

tasks: ['generate-report', 'resize-image', 'send-email']
processing: send-email
processing: resize-image
remaining : ['generate-report']


## 8. Sets — Unique, Unordered Collections

A set holds **unique** items. Great for tags, unique visitor IDs, etc.

- `SADD key member ...`
- `SMEMBERS key` — all members
- `SISMEMBER key member` — boolean check
- `SINTER`, `SUNION`, `SDIFF` — set algebra between keys

In [22]:
r.delete("post:42:tags", "post:43:tags")

r.sadd("post:42:tags", "redis", "python", "tutorial")
r.sadd("post:43:tags", "redis", "docker")

print("post 42 tags:", r.smembers("post:42:tags"))
print("Is 'python' a tag of post 42?", r.sismember("post:42:tags", "python"))

# Set operations
print("tags in BOTH posts :", r.sinter("post:42:tags", "post:43:tags"))
print("tags in EITHER post:", r.sunion("post:42:tags", "post:43:tags"))

post 42 tags: {'python', 'redis', 'tutorial'}
Is 'python' a tag of post 42? 1
tags in BOTH posts : {'redis'}
tags in EITHER post: {'python', 'docker', 'redis', 'tutorial'}


## 9. Sorted Sets — Leaderboards

Like a set, but every member has a **score** (a float), and members are kept ordered by score. This is what powers leaderboards, priority queues, time-ordered feeds, etc.

- `ZADD key score member`
- `ZRANGE key start stop [WITHSCORES]` — ascending
- `ZREVRANGE key start stop [WITHSCORES]` — descending (top-N)
- `ZINCRBY key amount member` — bump a score
- `ZRANK` / `ZREVRANK` — position of a member

In [23]:
r.delete("game:scores")

# Add players with their scores
r.zadd("game:scores", {"alice": 1500, "bob": 2300, "carol": 1800, "dave": 900})

# Player scores a goal
r.zincrby("game:scores", 250, "alice")

# Top 3 leaderboard (highest first)
print("Top 3:")
for rank, (player, score) in enumerate(
    r.zrevrange("game:scores", 0, 2, withscores=True), start=1
):
    print(f"  {rank}. {player:6s} {int(score)}")

# Where does Bob rank? (0 = top)
print("\nBob's rank:", r.zrevrank("game:scores", "bob"))

Top 3:
  1. bob    2300
  2. carol  1800
  3. alice  1750

Bob's rank: 0


## 10. Pub/Sub — Real-Time Messaging

Redis can act as a tiny message broker:
- **Publishers** send messages to a *channel*
- **Subscribers** receive messages on channels they're listening to

Below we run both sides in **one cell** using a background thread so it works inside a notebook.

In [24]:
import threading
import time

pubsub = r.pubsub()
pubsub.subscribe("news")

received = []

def listener():
    for message in pubsub.listen():
        if message["type"] == "message":
            received.append(message["data"])
            if message["data"] == "STOP":
                break

t = threading.Thread(target=listener, daemon=True)
t.start()

time.sleep(0.2)  # give the subscriber a moment to be ready

# Publisher sends a few messages
r.publish("news", "Redis 8 released!")
r.publish("news", "New tutorial posted")
r.publish("news", "STOP")

t.join(timeout=2)
pubsub.close()

print("Subscriber received:", received)

Subscriber received: ['Redis 8 released!', 'New tutorial posted', 'STOP']


## 11. Pipelines — Speed Up Many Commands

Each Redis command is a network round-trip. If you need to run many commands, batch them with a **pipeline**: all commands are sent at once and the responses come back together. Huge speedup for bulk work.

In [25]:
import time

# --- Without pipeline: 1000 round-trips ---
start = time.perf_counter()
for i in range(1000):
    r.set(f"bulk:{i}", i)
slow = time.perf_counter() - start

# Clean up before re-running
r.delete(*[f"bulk:{i}" for i in range(1000)])

# --- With pipeline: ~1 round-trip ---
start = time.perf_counter()
pipe = r.pipeline()
for i in range(1000):
    pipe.set(f"bulk:{i}", i)
pipe.execute()
fast = time.perf_counter() - start

print(f"without pipeline: {slow*1000:.1f} ms")
print(f"with pipeline   : {fast*1000:.1f} ms")
print(f"speedup         : {slow/fast:.1f}x")

# Cleanup
r.delete(*[f"bulk:{i}" for i in range(1000)])

without pipeline: 1269.4 ms
with pipeline   : 20.3 ms
speedup         : 62.5x


1000

## 12. Mini Project — Cache + Leaderboard

Let's tie it all together. We'll simulate fetching user profiles from a "slow database" and:

1. **Cache** results in Redis with a TTL (Strings + EX).
2. Track each profile's **view count** and show the **most-viewed users** (Sorted Set).

This is a real pattern used by countless web apps.

In [26]:
import json
import time

# --- Pretend "slow database" ---
FAKE_DB = {
    1: {"name": "Alice",  "country": "US"},
    2: {"name": "Bob",    "country": "UK"},
    3: {"name": "Carol",  "country": "DE"},
}

def fetch_from_db(user_id: int) -> dict:
    """Slow lookup that we want to cache."""
    time.sleep(0.5)            # simulate latency
    return FAKE_DB[user_id]

def get_user(user_id: int) -> dict:
    """Cache-aside pattern: try Redis first, fall back to DB."""
    cache_key = f"user:{user_id}:profile"

    cached = r.get(cache_key)
    if cached is not None:
        source = "cache"
        user = json.loads(cached)
    else:
        source = "db"
        user = fetch_from_db(user_id)
        # Cache the result for 60 seconds
        r.set(cache_key, json.dumps(user), ex=60)

    # Track popularity in a sorted set
    r.zincrby("popular:users", 1, user_id)

    print(f"  [{source}] user {user_id} -> {user}")
    return user

# Reset state for a clean demo
r.delete("popular:users")
for k in r.scan_iter("user:*:profile"):
    r.delete(k)

# Simulate traffic
print("First reads (slow, hit DB):")
t0 = time.perf_counter(); get_user(1); get_user(2); get_user(1)
print(f"  elapsed: {(time.perf_counter()-t0)*1000:.0f} ms\n")

print("Repeat reads (fast, hit cache):")
t0 = time.perf_counter()
for _ in range(3): get_user(1)
for _ in range(2): get_user(2)
for _ in range(5): get_user(3)
print(f"  elapsed: {(time.perf_counter()-t0)*1000:.0f} ms\n")

print("Most-viewed users:")
for uid, views in r.zrevrange("popular:users", 0, -1, withscores=True):
    print(f"  user {uid}: {int(views)} views")

First reads (slow, hit DB):
  [db] user 1 -> {'name': 'Alice', 'country': 'US'}
  [db] user 2 -> {'name': 'Bob', 'country': 'UK'}
  [cache] user 1 -> {'name': 'Alice', 'country': 'US'}
  elapsed: 1015 ms

Repeat reads (fast, hit cache):
  [cache] user 1 -> {'name': 'Alice', 'country': 'US'}
  [cache] user 1 -> {'name': 'Alice', 'country': 'US'}
  [cache] user 1 -> {'name': 'Alice', 'country': 'US'}
  [cache] user 2 -> {'name': 'Bob', 'country': 'UK'}
  [cache] user 2 -> {'name': 'Bob', 'country': 'UK'}
  [db] user 3 -> {'name': 'Carol', 'country': 'DE'}
  [cache] user 3 -> {'name': 'Carol', 'country': 'DE'}
  [cache] user 3 -> {'name': 'Carol', 'country': 'DE'}
  [cache] user 3 -> {'name': 'Carol', 'country': 'DE'}
  [cache] user 3 -> {'name': 'Carol', 'country': 'DE'}
  elapsed: 540 ms

Most-viewed users:
  user 3: 5 views
  user 1: 5 views
  user 2: 3 views


## 13. Cleanup & Cheat Sheet

### Useful admin commands
| Command       | What it does                          |
|---------------|---------------------------------------|
| `KEYS *`      | List all keys (avoid in production!)  |
| `SCAN 0`      | Iterate keys safely                   |
| `DBSIZE`      | Number of keys in current DB          |
| `FLUSHDB`     | ⚠️ Wipe current database              |
| `FLUSHALL`    | ⚠️ Wipe ALL databases                 |
| `INFO`        | Server stats                          |
| `MONITOR`     | Watch every command live (debug)      |

The cell below cleans up keys we created in this notebook.

In [27]:
# Delete just the keys this notebook created (safer than FLUSHDB)
patterns = ["greeting", "page:views", "temp:otp", "user:*", "tasks",
            "post:*", "game:scores", "popular:users", "bulk:*"]

deleted = 0
for pattern in patterns:
    for key in r.scan_iter(pattern):
        r.delete(key)
        deleted += 1

print(f"Deleted {deleted} keys.")
print("DB size now:", r.dbsize())

Deleted 11 keys.
DB size now: 0



 **To Try-out**

1. **Streams** (`XADD`, `XREAD`) — durable, multi-consumer messaging (replaces Pub/Sub for reliable delivery).
2. **Persistence** — RDB snapshots vs AOF (append-only file).
3. **Replication & High Availability** — primary/replica, **Redis Sentinel**.
4. **Sharding** — **Redis Cluster** for horizontal scaling.
5. **Lua scripting / Functions** — server-side atomic logic.
6. **Modules**: RedisJSON, RediSearch, RedisTimeSeries, Redis Vector Search.

### 📚 Recommended resources
- Official docs: https://redis.io/docs/latest/
- Interactive tutorial: https://try.redis.io
- Free courses: https://university.redis.io
- Command reference: https://redis.io/commands/

### ✅ Practice ideas
- Build a **rate limiter** (`INCR` + `EXPIRE`).
- Build a **session store** for a Flask/FastAPI app.
- Build a **chat room** using Pub/Sub or Streams.
- Add a **Redis cache** in front of a SQL query in your own project.